# L02 · CE·엔트로피·KL·KD

## Goal

**예상 시간:** 40분 · **경로:** fast, full

- CE·entropy·KL을 계산한다
- KL 방향을 말로 설명한다
- JSD beta 경계를 확인한다

### 현재 위치: L01 → **L02** → L03

```text
Prompt/Data -> state source -> ... -> L02 -> ... -> fair evaluation
```

Alt text: The course map highlights L02 between its prerequisite and next lesson; every method remains connected to the same evaluation stage.

## Setup

In [1]:
LESSON_ID = "L02"
from pathlib import Path
import sys
import torch

repo_root = Path.cwd()
if not (repo_root / "src").exists():
    repo_root = Path.cwd().parents[1]
sys.path.insert(0, str(repo_root / "src"))

import opd_study
from opd_study.device import resolve_device
from opd_study.utils import seed_everything

seed_everything(42)
device_report = resolve_device("cpu")
print({"lesson": LESSON_ID, "opd_study": opd_study.__version__,
       "torch": torch.__version__, "device": device_report.selected,
       "profile": "toy", "network": "not required"})

{'lesson': 'L02', 'opd_study': '0.1.0.dev0', 'torch': '2.13.0', 'device': 'cpu', 'profile': 'toy', 'network': 'not required'}


## Steps

### 1/3 · 8–12 min

KL은 대칭 거리가 아니다. 이 강좌는 forward를 KL(teacher||student), reverse를 KL(student||teacher)로 고정하고 함수 인자에도 두 분포 이름을 쓴다.

그림 대체 설명: 출력의 label과 숫자는 색 없이도 읽을 수 있다.

### 핵심 원리

hard-label CE는 정답 token 하나만 남기지만 KD는 teacher의 전체 분포를 보존한다. `KL(teacher || student)`는 teacher가 확률을 둔 곳을 빠뜨릴 때 크게 벌하고, `KL(student || teacher)`는 student가 teacher의 저확률 영역에 질량을 둘 때 크게 벌한다. 그래서 전자는 coverage-seeking, 후자는 mode-seeking 경향으로 설명되지만 이는 보장된 행동 법칙이 아니다.

temperature `T`는 logits를 `T`로 나눠 분포를 부드럽게 한다. 고전 KD는 gradient 크기 변화를 보정하려 `T²`를 곱한다. 이 레포는 모든 log-softmax를 float32에서 계산하고 빈 mask·0 이하 temperature를 즉시 거부한다.

### 실제 구현: 왜 이렇게 만들었나

모든 KL API는 `(teacher_logits, student_logits)`라는 이름을 강제한다. reduction 전 shape는 `[B,T]`; 마지막에는 response mask로만 평균낸다. generalized JSD의 beta 경계는 별도 branch로 계산해 `log(0)`을 피한다.

실제 코드: [`math.py`](../../src/opd_study/math.py), [`losses.py`](../../src/opd_study/algorithms/losses.py).

In [2]:
import inspect
from opd_study.math import forward_kl_from_logits, reverse_kl_from_logits

objects_to_show = (forward_kl_from_logits, reverse_kl_from_logits,)
for object_to_show in objects_to_show:
    source_lines = inspect.getsource(object_to_show).splitlines()
    print(f"\n# {object_to_show.__module__}.{object_to_show.__qualname__}")
    print("\n".join(source_lines[:80]))
    if len(source_lines) > 80:
        print(f"... {len(source_lines) - 80} more lines; open the linked source file")


# opd_study.math.forward_kl_from_logits
def forward_kl_from_logits(
    teacher_logits: Tensor,
    student_logits: Tensor,
    *,
    temperature: float = 1.0,
) -> Tensor:
    """Return ``KL(teacher || student)`` without reducing token/batch dimensions."""

    _validate_logits(teacher_logits, student_logits)
    teacher_log_p = log_probs(teacher_logits, temperature=temperature)
    student_log_p = log_probs(student_logits, temperature=temperature)
    return (teacher_log_p.exp() * (teacher_log_p - student_log_p)).sum(dim=-1)

# opd_study.math.reverse_kl_from_logits
def reverse_kl_from_logits(
    teacher_logits: Tensor,
    student_logits: Tensor,
    *,
    temperature: float = 1.0,
) -> Tensor:
    """Return ``KL(student || teacher)`` without reducing token/batch dimensions."""

    _validate_logits(teacher_logits, student_logits)
    teacher_log_p = log_probs(teacher_logits, temperature=temperature)
    student_log_p = log_probs(student_logits, temperature=temperature)
    retur

### 다른 선택지는 없나?

full logits가 비싸면 top-k logits나 sampled token만 저장할 수 있다. top-k는 tail mass를 버리므로 retained probability mass와 근사 오차를 같이 보고한다. API teacher가 log-prob을 일부만 주면 full-KL 구현으로 가장하지 않는다.

### 2/3 · 실행하고 관찰하기

실행 전 예측: L02의 첫 출력에서 가장 먼저 확인해야 할 invariant는 무엇일까? 한 문장으로 적고 실행한다.

In [3]:
from opd_study.math import (entropy_from_logits, forward_kl_from_logits,
                            generalized_jsd_from_logits, reverse_kl_from_logits)

teacher = torch.log(torch.tensor([[0.70, 0.20, 0.10]]))
student = torch.log(torch.tensor([[0.40, 0.35, 0.25]]))
values = {
    "H(teacher)": entropy_from_logits(teacher).item(),
    "KL(teacher||student)": forward_kl_from_logits(teacher, student).item(),
    "KL(student||teacher)": reverse_kl_from_logits(teacher, student).item(),
    "JSD_beta=.5": generalized_jsd_from_logits(teacher, student, beta=.5).item(),
}
print({name: round(value, 5) for name, value in values.items()})

{'H(teacher)': 0.80182, 'KL(teacher||student)': 0.18818, 'KL(student||teacher)': 0.20109, 'JSD_beta=.5': 0.04768}


In [4]:
boundaries = [generalized_jsd_from_logits(teacher, student, beta=beta).item()
              for beta in (0.0, 0.25, 0.5, 0.75, 1.0)]
print("beta sweep:", [round(value, 5) for value in boundaries])
print("Argument order is part of the definition; KL is not symmetric.")

beta sweep: [0.18818, 0.03537, 0.04768, 0.0365, 0.20109]
Argument order is part of the definition; KL is not symmetric.


## Checks

In [5]:
assert values["KL(teacher||student)"] >= 0
assert values["KL(student||teacher)"] >= 0
assert abs(boundaries[0] - values["KL(teacher||student)"]) < 1e-6
assert abs(boundaries[-1] - values["KL(student||teacher)"]) < 1e-6
print("check passed: non-negativity and named beta boundaries")

check passed: non-negativity and named beta boundaries


**연습 (7분):** teacher `[0.99,0.01]`, student `[0.5,0.5]`에서 FKL과 RKL을 손으로 계산하고 두 인자를 바꿨을 때 값이 왜 달라지는지 설명하라.

<details><summary>확인 기준</summary>`sum teacher*(log teacher-log student)`와 `sum student*(log student-log teacher)`를 따로 계산하고 KL 비대칭을 언급한다.</details>

## 내가 자주 틀리는 것

### M1 — `KL(p,q)`의 방향을 암기만 하기

- 틀린 형태: 함수 인자 이름 없이 첫/둘째 인자를 추측한다.
- 왜 틀렸나: 논문·라이브러리 convention이 다르다.
- 고친 형태: `KL(teacher || student)`처럼 분포 역할을 쓴다.
- 관련 검사: `test_forward_kl_matches_hand_calculation`

### M2 — temperature만 바꾸고 gradient scale을 비교하기

- 틀린 형태: `T²` 보정 없이 loss 크기 차이를 objective 우열로 읽는다.
- 왜 틀렸나: softmax derivative scale도 변한다.
- 고친 형태: 동일 convention과 보정 여부를 run card에 기록한다.
- 관련 검사: `test_temperature_and_empty_masks_fail_loudly`

## 60초 요약

1. CE·entropy·KL을 계산한다
2. KL 방향을 말로 설명한다
3. JSD beta 경계를 확인한다

## Next Steps

다음 노트북으로 가기 전, 위 assertion을 다시 실행하고 틀린 예측 한 줄을 남긴다.

### Sources

- [`gkd`](https://arxiv.org/abs/2306.13649v3) · `2306.13649v3` · license `CC-BY-4.0` · [audited manifest](../../docs/sources.yml)